# Vertex-wise pMTG Brain–Behavior Analysis Without SES Residualization

This notebook correlates every pMTG vertex-to-network FC value with cognitive EFA factors and income-to-needs ratio (INR). FC is residualized for sex, age, scanner software, and motion; cognitive EFA factors are loaded from the matched exploratory cognitive-factor workflow. INR remains a tested SES outcome.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import seaborn as sns

from scipy import stats
from sklearn.linear_model import LinearRegression
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='colorblind')


In [ ]:
# Notebook-local analysis helpers.
STANDARD_FC_COVARIATES = [
    'demo_sex_v2',
    'interview_age',
    'site_id_l',
    'ehi1b',
    'mean_fd_0.20',
]

DEFAULT_CATEGORICAL_COVARIATES = {
    'demo_sex_v2',
    'site_id_l',
    'ehi1b',
}


def standardize_subject_id(subject_ids):
    return (
        subject_ids.astype(str)
        .str.replace('_', '', regex=False)
        .str.replace('^sub-', '', regex=True)
    )


def load_motion_qa(motion_qa_path, motion_column='mean_fd_0.20'):
    motion_qa = pd.read_csv(motion_qa_path)
    motion_qa = motion_qa[['src_subject_id', motion_column]].copy()
    motion_qa['src_subject_id'] = standardize_subject_id(motion_qa['src_subject_id'])
    motion_qa = motion_qa.drop_duplicates(subset=['src_subject_id'])
    motion_qa[motion_column] = pd.to_numeric(motion_qa[motion_column], errors='coerce')
    return motion_qa


def merge_motion_qa(df, motion_qa_path, motion_column='mean_fd_0.20', how='left'):
    motion_qa = load_motion_qa(motion_qa_path, motion_column=motion_column)

    merged = df.copy()
    merged['src_subject_id'] = standardize_subject_id(merged['src_subject_id'])
    if motion_column in merged.columns:
        existing_motion = merged.groupby('src_subject_id')[motion_column].first()
        merged = merged.drop(columns=[motion_column])
        merged = merged.merge(motion_qa, on='src_subject_id', how=how)
        merged[motion_column] = merged[motion_column].fillna(
            merged['src_subject_id'].map(existing_motion)
        )
    else:
        merged = merged.merge(motion_qa, on='src_subject_id', how=how)
    return merged


def complete_covariate_mask(df, covariates, categorical_covariates=DEFAULT_CATEGORICAL_COVARIATES):
    categorical_covariates = set(categorical_covariates)
    complete = pd.Series(True, index=df.index)

    for covariate in covariates:
        is_categorical = df[covariate].dtype == 'object' or covariate in categorical_covariates
        if is_categorical:
            complete &= df[covariate].notna()
        else:
            complete &= pd.to_numeric(df[covariate], errors='coerce').notna()

    return complete

def encode_regression_covariates(df, covariates, categorical_covariates=DEFAULT_CATEGORICAL_COVARIATES):
    encoded_parts = []
    categorical_covariates = set(categorical_covariates)

    for covariate in covariates:
        if df[covariate].dtype == 'object' or covariate in categorical_covariates:
            encoded_parts.append(
                pd.get_dummies(
                    df[covariate],
                    prefix=covariate,
                    drop_first=True,
                    dtype=float,
                )
            )
        else:
            encoded_parts.append(
                pd.to_numeric(df[covariate], errors='coerce').to_frame(covariate)
            )

    if not encoded_parts:
        return pd.DataFrame(index=df.index)
    return pd.concat(encoded_parts, axis=1)


def residualize_fc_profiles(df, fc_columns, covariates=STANDARD_FC_COVARIATES):
    df = df.copy()
    covariate_complete = complete_covariate_mask(df, covariates)

    validity_groups = {}
    for column in fc_columns:
        valid_idx = df[column].notnull() & covariate_complete
        key = valid_idx.to_numpy(dtype=np.bool_).tobytes()
        if key not in validity_groups:
            validity_groups[key] = (valid_idx, [])
        validity_groups[key][1].append(column)

    residual_frames = []
    for valid_idx, columns in validity_groups.values():
        output_columns = [column + '_resid' for column in columns]
        residuals = pd.DataFrame(np.nan, index=df.index, columns=output_columns)
        if valid_idx.sum() > 0:
            observed = df.loc[valid_idx, columns].to_numpy()
            covariate_matrix = encode_regression_covariates(
                df.loc[valid_idx],
                covariates,
            )
            if covariate_matrix.shape[1] == 0:
                predicted = np.tile(observed.mean(axis=0), (len(observed), 1))
            else:
                model = LinearRegression()
                model.fit(covariate_matrix, observed)
                predicted = model.predict(covariate_matrix)
            residuals.loc[valid_idx, output_columns] = observed - predicted
        residual_frames.append(residuals)

    if residual_frames:
        df = pd.concat([df, *residual_frames], axis=1)

    return df


def parse_vertexwise_fc_column(column):
    if not column.endswith('_fz'):
        return None

    parts = column[:-3].rsplit('_', 2)
    if len(parts) != 3:
        return None
    network, vertex, hemisphere = parts
    if not network or not vertex.isdigit() or hemisphere not in {'L', 'R'}:
        return None
    return network, int(vertex), hemisphere


def get_vertexwise_fc_columns(df):
    # Analysis selectors always exclude generated full-network FC columns.
    columns = []
    for column in df.columns:
        parsed = parse_vertexwise_fc_column(column)
        if parsed is None:
            continue
        network, _, _ = parsed
        if network.endswith('_full'):
            continue
        columns.append(column)
    return columns


def compute_correlations(df, measures, fc_columns, required_nonmissing=None):
    results = []
    required_nonmissing = list(required_nonmissing or [])

    for measure in measures:
        for column in fc_columns:
            analysis_columns = list(dict.fromkeys([column, measure, *required_nonmissing]))
            temp_df = df[analysis_columns].dropna()

            if (
                len(temp_df) < 2
                or temp_df[column].nunique(dropna=True) < 2
                or temp_df[measure].nunique(dropna=True) < 2
            ):
                r_value = np.nan
                p_value = np.nan
            else:
                r_value, p_value = stats.pearsonr(temp_df[column], temp_df[measure])

            results.append({
                'measure': measure,
                'col': column,
                'r': r_value,
                'p': p_value,
                'n': len(temp_df),
            })

    return pd.DataFrame(results)


def apply_multiple_comparison_corrections(results_df, alpha=0.05, fdr_group_col=None):
    if results_df.empty:
        return results_df.assign(
            p_bonf=pd.Series(dtype=float),
            sig_bonf=pd.Series(dtype=bool),
            p_fdr=pd.Series(dtype=float),
            sig_fdr=pd.Series(dtype=bool),
        )

    corrected = results_df.copy()
    n_tests = len(corrected)
    corrected['p_bonf'] = np.minimum(corrected['p'] * n_tests, 1.0)
    corrected['sig_bonf'] = corrected['p_bonf'] < alpha

    valid_p = corrected['p'].notna()
    corrected['p_fdr'] = np.nan
    corrected['sig_fdr'] = False

    if fdr_group_col is None:
        fdr_families = pd.Series('all', index=corrected.index)
    else:
        fdr_families = corrected[fdr_group_col].astype('string').fillna('<missing>')

    for family in fdr_families.unique():
        family_valid_p = valid_p & fdr_families.eq(family)
        if family_valid_p.any():
            reject, p_fdr, _, _ = multipletests(
                corrected.loc[family_valid_p, 'p'],
                alpha=alpha,
                method='fdr_bh',
            )
            corrected.loc[family_valid_p, 'p_fdr'] = p_fdr
            corrected.loc[family_valid_p, 'sig_fdr'] = reject

    return corrected


## Configuration

Update the three input paths before running. Generated `*_full` network summaries are excluded from the analysis table before testing.

In [ ]:
RESIDUALIZE_FOR_SES = False
SES_VARIABLE = 'inr'
SES_RESIDUALIZATION_COVARIATE = 'inr'
WRITE_SIGNIFICANT_CIFTI_MAPS = True
MAX_RESULT_PLOTS = 20

VERBAL_ABILITY_DIR = Path('/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability')
LOCAL_DATA_DIR = VERBAL_ABILITY_DIR / 'Final'
WRANGLED_DATA_PATH = LOCAL_DATA_DIR / 'wrangled_pMTG_FC_data_midb61_meanFC.csv'
PHENOTYPE_PATH = LOCAL_DATA_DIR / 'midb61_meanFC_clusters_motion_resid_2026-07-14_17-56.csv'
VERTEX_FC_PATH = LOCAL_DATA_DIR / 'pMTG_FC_profiles_midb61_vertexwiseFC.csv'
MOTION_QA_PATH = LOCAL_DATA_DIR / 'motion_QA_results.csv'
PMTG_TEMPLATE_PATH = Path('/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability/pMTG_regions.dtseries.nii')
OUTPUT_DIR = LOCAL_DATA_DIR / 'vertexwise_brain_behavior_results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PCA_RESULTS_DIR = LOCAL_DATA_DIR / 'pca_results'

ANALYSIS_SLUG = 'without_ses_residualization'
EFA_SCORE_MODEL_NAME = ANALYSIS_SLUG
EFA_SCORE_PREFIX = 'ses_cognitive' if RESIDUALIZE_FOR_SES else 'no_ses_cognitive'
EFA_SCORE_PATH = PCA_RESULTS_DIR / f'efa_cognitive_scores_{EFA_SCORE_MODEL_NAME}.csv'


## Load and merge vertex-wise FC with phenotypes

In [ ]:
phenotype_columns = [
    'src_subject_id',
    'demo_comb_income_v2',
    'demo_roster_v2',
    'demo_sex_v2',
    'interview_age',
    'site_id_l',
    'ehi1b',
    SES_VARIABLE,
    'inr_missing',
    'poverty_line_2017',
]
phenotypes = pd.read_csv(PHENOTYPE_PATH, usecols=lambda column: column in phenotype_columns)
phenotypes = merge_motion_qa(phenotypes, MOTION_QA_PATH, how='left')
phenotypes['src_subject_id'] = standardize_subject_id(phenotypes['src_subject_id'])

vertex_fc = pd.read_csv(VERTEX_FC_PATH)
if 'subject_id' in vertex_fc.columns:
    vertex_fc = vertex_fc.rename(columns={'subject_id': 'src_subject_id'})
vertex_fc['src_subject_id'] = standardize_subject_id(vertex_fc['src_subject_id'])

for name, table in {'phenotype': phenotypes, 'vertex FC': vertex_fc}.items():
    duplicated = table['src_subject_id'].duplicated().sum()

df = phenotypes.merge(vertex_fc, on='src_subject_id', how='inner', validate='one_to_one')

# Drop generated full-network Fisher-z FC columns before any analysis selectors run.
full_vertex_fc_cols = [column for column in df.columns if '_fz' in str(column) and '_full' in str(column)]
if full_vertex_fc_cols:
    df = df.drop(columns=full_vertex_fc_cols)
print(f'Dropped {len(full_vertex_fc_cols)} generated full-network vertexwise FC columns.')

required_inr_cols = [SES_VARIABLE, 'inr_missing', 'poverty_line_2017'] if RESIDUALIZE_FOR_SES else []
missing_inr_cols = [col for col in required_inr_cols if col not in df.columns]
if RESIDUALIZE_FOR_SES and missing_inr_cols:
    wrangled_inr_columns = [
        'src_subject_id',
        'demo_comb_income_v2',
        'demo_roster_v2',
        *required_inr_cols,
    ]
    wrangled_inr = pd.read_csv(
        WRANGLED_DATA_PATH,
        usecols=lambda column: column in wrangled_inr_columns,
    )
    wrangled_inr['src_subject_id'] = standardize_subject_id(wrangled_inr['src_subject_id'])
    wrangled_inr = wrangled_inr.drop_duplicates(subset='src_subject_id')

    existing_inr_cols = [col for col in wrangled_inr.columns if col != 'src_subject_id' and col in df.columns]
    df = df.drop(columns=existing_inr_cols)
    df = df.merge(wrangled_inr, on='src_subject_id', how='left', validate='many_to_one')

missing_inr_cols = [col for col in required_inr_cols if col not in df.columns]


efa_scores = pd.read_csv(EFA_SCORE_PATH)
efa_scores['src_subject_id'] = standardize_subject_id(efa_scores['src_subject_id'])
efa_scores = efa_scores.drop_duplicates(subset='src_subject_id')
cognitive_efa_measures = [
    col for col in efa_scores.columns if col.startswith(f'{EFA_SCORE_PREFIX}_EF')
]

pre_merge_rows = len(df)
df = df.drop(columns=[col for col in cognitive_efa_measures if col in df.columns])
df = df.merge(
    efa_scores[['src_subject_id', *cognitive_efa_measures]],
    on='src_subject_id',
    how='left',
)

vertex_fc_cols = get_vertexwise_fc_columns(df)
missing_covariates = [covariate for covariate in STANDARD_FC_COVARIATES if covariate not in df.columns]

for measure in cognitive_efa_measures:
    df[measure] = pd.to_numeric(df[measure], errors='coerce')

print(f'Loaded {len(df):,} subjects with {len(vertex_fc_cols):,} vertexwise FC columns.')
print(f'Cognitive EFA measures: {cognitive_efa_measures}')
if RESIDUALIZE_FOR_SES:
    print(f'Missing raw INR values excluded from SES-residualized correlations: {df[SES_VARIABLE].isna().sum()}')
else:
    print('INR is not required for this no-SES vertexwise brain-behavior analysis.')


## Residualize FC and cognitive EFA factors

The wide FC residualizer fits columns sharing a missing-data pattern together, making the vertex-wise calculation much faster than fitting thousands of separate models.

In [ ]:
fc_covariates = list(STANDARD_FC_COVARIATES)
if RESIDUALIZE_FOR_SES:
    fc_covariates.append(SES_RESIDUALIZATION_COVARIATE)

print('FC covariates:', fc_covariates)
df = residualize_fc_profiles(df, vertex_fc_cols, covariates=fc_covariates)

analysis_fc_cols = [f'{column}_resid' for column in vertex_fc_cols]
for measure in cognitive_efa_measures:
    df[measure] = pd.to_numeric(df[measure], errors='coerce')

analysis_measures = cognitive_efa_measures.copy()

print(f'Analysis FC columns: {len(analysis_fc_cols)}')
print('Analysis cognitive EFA measures:', analysis_measures)
for measure in analysis_measures:
    measure_sample = df[measure].notna().sum()
    print(f'Sample for {measure}: {measure_sample}')


## Vertex-wise brain–behavior correlations

In [ ]:
results_df = compute_correlations(
    df,
    analysis_measures,
    analysis_fc_cols,
    required_nonmissing=[SES_VARIABLE] if RESIDUALIZE_FOR_SES else None,
)
results_df = apply_multiple_comparison_corrections(results_df, alpha=0.05).drop(
    columns=['p_bonf', 'sig_bonf'],
    errors='ignore',
)

results_df['source_fc_column'] = results_df['col'].str.removesuffix('_resid')
parsed = results_df['source_fc_column'].map(parse_vertexwise_fc_column)
results_df[['network', 'python_index', 'hemisphere']] = pd.DataFrame(
    parsed.tolist(),
    index=results_df.index,
)
results_df['abs_r'] = results_df['r'].abs()
results_df = results_df.sort_values(['p_fdr', 'p', 'abs_r'], ascending=[True, True, False])
cognitive_results_df = results_df.copy()

all_results_path = OUTPUT_DIR / f'vertexwise_brain_behavior_{ANALYSIS_SLUG}_all.csv'
cognitive_results_path = OUTPUT_DIR / f'vertexwise_brain_behavior_{ANALYSIS_SLUG}_cognitive_efa_all.csv'
cognitive_significant_path = OUTPUT_DIR / f'vertexwise_brain_behavior_{ANALYSIS_SLUG}_cognitive_efa_fdr_significant.csv'
results_df.to_csv(all_results_path, index=False)
cognitive_results_df.to_csv(cognitive_results_path, index=False)
cognitive_results_df[cognitive_results_df['sig_fdr']].to_csv(cognitive_significant_path, index=False)

print(f'Cognitive EFA-FC tests: {len(cognitive_results_df):,}')
print(f'Cognitive EFA FDR-significant tests: {cognitive_results_df["sig_fdr"].sum():,}')
print(f'Sample-size range: {results_df["n"].min()}-{results_df["n"].max()}')
print(f'Saved all results to {all_results_path}')
print(f'Saved cognitive EFA results to {cognitive_results_path}')
print(f'Saved FDR-significant cognitive EFA results to {cognitive_significant_path}')
display(results_df.head(25))


## Significant-result summaries

In [ ]:
def summarize_fdr_results(label, output_slug, family_results):
    print()
    print(label)
    if family_results.empty:
        print('Not tested in this analysis.')
        return family_results.copy()

    significant = family_results[family_results['sig_fdr']].copy()
    if significant.empty:
        print('No correlations survived FDR correction within this cognitive EFA family.')
        return significant

    summary = (
        significant.groupby(['measure', 'network', 'hemisphere'], as_index=False)
        .agg(n_significant_vertices=('python_index', 'nunique'), max_abs_r=('abs_r', 'max'))
        .sort_values(['n_significant_vertices', 'max_abs_r'], ascending=False)
    )
    summary_path = OUTPUT_DIR / f'vertexwise_brain_behavior_{ANALYSIS_SLUG}_{output_slug}_summary.csv'
    summary.to_csv(summary_path, index=False)
    print(f'Saved summary to {summary_path}')
    display(summary)

    plot_groups = list(significant.groupby(['measure', 'network', 'hemisphere']))[:MAX_RESULT_PLOTS]
    for (measure, network, hemisphere), group in plot_groups:
        background = family_results[
            (family_results['measure'] == measure)
            & (family_results['network'] == network)
            & (family_results['hemisphere'] == hemisphere)
        ]
        fig, ax = plt.subplots(figsize=(10, 3.5))
        ax.scatter(background['python_index'], background['r'], s=14, alpha=0.35, label='Not FDR significant')
        ax.scatter(group['python_index'], group['r'], s=28, color='crimson', label='FDR significant')
        ax.axhline(0, color='black', linewidth=0.8)
        ax.set(title=f'{measure}: {network}, {hemisphere} pMTG', xlabel='Zero-based Python dense index', ylabel='Pearson r')
        ax.legend(frameon=False)
        plt.tight_layout()
        plt.show()

    return significant


cognitive_significant = summarize_fdr_results(
    'Cognitive EFA factor results (FDR corrected)',
    'cognitive_efa_fdr',
    cognitive_results_df,
)


## Optional CIFTI maps of FDR-significant cognitive EFA effects

When enabled, this writes one scalar CIFTI per FDR-significant cognitive EFA factor/network pair. Values are Pearson correlations at pMTG vertices and zero elsewhere.

In [ ]:
significant_by_family = {
    'cognitive_efa': cognitive_significant,
}
has_significant_maps = any(not family_results.empty for family_results in significant_by_family.values())

if WRITE_SIGNIFICANT_CIFTI_MAPS and has_significant_maps:
    map_template_path = Path('/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability/pMTG_regions.dtseries.nii')
    map_output_root = Path(f'/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability/Final/vertexwise_brain_behavior_results/maps_{ANALYSIS_SLUG}')
    template = nib.load(map_template_path)
    brain_axis = template.header.get_axis(1)

    for family_name, significant in significant_by_family.items():
        if significant.empty:
            continue
        map_output_dir = map_output_root / family_name
        map_output_dir.mkdir(parents=True, exist_ok=True)
        for (measure, network), group in significant.groupby(['measure', 'network']):
            data = np.zeros(len(brain_axis), dtype=np.float32)
            data[group['python_index'].astype(int).to_numpy()] = group['r'].to_numpy(dtype=np.float32)
            map_name = f'{measure}__{network}'
            scalar_axis = nib.cifti2.cifti2_axes.ScalarAxis([map_name])
            header = nib.Cifti2Header.from_axes((scalar_axis, brain_axis))
            output_img = nib.Cifti2Image(data[None, :], header=header, nifti_header=template.nifti_header)
            safe_name = map_name.replace('/', '-').replace(' ', '_')
            nib.save(output_img, map_output_dir / f'{safe_name}.dscalar.nii')
        print(f'Saved {family_name} significant-effect CIFTIs to {map_output_dir}')
else:
    print('CIFTI export disabled or no FDR-significant cognitive EFA results were found.')


## Selected Network EFA Factor Maps

For each cognitive EFA factor, write a colored CIFTI label map marking vertices with FDR-significant brain-behavior correlations in DMN, DAN, AMN, and FPN. Single-network vertices use the network color; overlap vertices use the average of the overlapping network colors because a CIFTI label can store only one color per vertex.

In [ ]:
from itertools import combinations  # Enumerate every selected-network overlap combination.
from matplotlib.colors import to_rgb  # Convert network hex colors to CIFTI-compatible RGB values.

selected_network_display_order = ['DMN', 'DAN', 'AMN', 'FPN']  # Set the plotted order for single-network and overlap labels.
selected_network_source_labels = {  # Map display names to the source network labels in vertexwise FC columns.
    'DMN': 'DMN',  # Treat source label 'DMN' as display network 'DMN'.
    'DAN': 'DAN',  # Treat source label 'DAN' as display network 'DAN'.
    'AMN': 'CO',  # Treat source label 'CO' as display network 'AMN'.
    'FPN': 'FP',  # Treat source label 'FP' as display network 'FPN'.
}
network_colors = {  # Store exact display colors used for selected-network map labels.
    'DMN': '#fb2e2e',  # Use the established network color for DMN.
    'DAN': '#2efe2e',  # Use the established network color for DAN.
    'AMN': '#8a2edb',  # Use the established network color for AMN.
    'FPN': '#ffff2e',  # Use the established network color for FPN.
    'FP': '#ffff2e',  # Use the established network color for FP.
    'LANG': '#40cce9',  # Use the established network color for LANG.
    'VAN': '#40cce9',  # Use the established network color for VAN.
}
network_color_lookup = network_colors.copy()  # Start color lookup from the exact network_colors table.
network_color_lookup['FPN'] = network_colors['FP']  # Make FPN use the exact FP color from network_colors.


def exact_network_rgba(network):  # Convert one network color to an RGBA tuple without averaging.
    rgb = to_rgb(network_color_lookup[network])  # Read the exact hex color for this network.
    return tuple(rgb + (1.0,))  # Add full opacity for the CIFTI label table.


def base_vertexwise_network(network):  # Strip hemisphere/full suffixes from vertexwise network labels.
    for suffix in ('_left', '_right', '_full'):  # Check every suffix used by the vertexwise FC columns.
        if network.endswith(suffix):  # Detect whether the current network label has this suffix.
            return network[: -len(suffix)]  # Return the network name before the suffix.
    return network  # Return unchanged labels that already contain only the base network.


def selected_display_network(network):  # Convert a source FC network label into a selected display network.
    source_network = base_vertexwise_network(network)  # Normalize the network label before matching.
    for display_network, source_label in selected_network_source_labels.items():  # Test each selected display/source pair.
        if source_network == source_label:  # Keep rows whose source label belongs to a selected network.
            return display_network  # Return the display label used in maps and summaries.
    return None  # Drop networks outside the selected-network set.


def selected_network_tuple(values):  # Build an ordered tuple of selected networks present at one vertex.
    observed_networks = set(values)  # Collapse repeated network hits at the same vertex.
    return tuple(  # Preserve selected_network_display_order for stable label values and legends.
        network  # Keep this network in the tuple when the vertex has a hit for it.
        for network in selected_network_display_order  # Walk through the canonical selected-network order.
        if network in observed_networks  # Include only networks actually observed at this vertex.
    )


def label_color_for_combo(combo):  # Assign one RGBA color to a single-network or overlap label.
    if len(combo) == 1:  # Use exact colors for vertices related to only one selected network.
        return exact_network_rgba(combo[0])  # Return the unmodified network_colors value as RGBA.
    rgb_values = np.array([to_rgb(network_color_lookup[network]) for network in combo])  # Convert overlap colors to RGB rows.
    return tuple(rgb_values.mean(axis=0).tolist() + [1.0])  # Average only true overlaps and add full opacity.


combo_label_table = {0: ('not_significant_selected_networks', (0.0, 0.0, 0.0, 0.0))}  # Initialize label 0 as transparent background.
combo_to_label_value = {}  # Store each network combination's integer label value.
next_label_value = 1  # Start significant selected-network labels at 1.
for combo_size in range(1, len(selected_network_display_order) + 1):  # Enumerate singleton through full-overlap labels.
    for combo in combinations(selected_network_display_order, combo_size):  # Generate combinations of this size.
        combo = tuple(combo)  # Store combinations as immutable dictionary keys.
        combo_to_label_value[combo] = next_label_value  # Assign this combination its label value.
        combo_label_table[next_label_value] = ('+'.join(combo), label_color_for_combo(combo))  # Store label name and RGBA color.
        next_label_value += 1  # Advance the next available CIFTI label value.

for network in selected_network_display_order:  # Verify single-network labels use exact network colors.
    singleton_label_value = combo_to_label_value[(network,)]  # Look up this network's singleton label value.
    singleton_label_name, singleton_rgba = combo_label_table[singleton_label_value]  # Read the singleton label entry.

selected_factor_map_root = OUTPUT_DIR / f'selected_network_efa_factor_maps_{ANALYSIS_SLUG}'  # Choose the output folder for selected-network maps.
selected_factor_map_root.mkdir(parents=True, exist_ok=True)  # Create the map output folder if needed.
selected_factor_summary_rows = []  # Collect one summary row per EFA factor and network combination.

map_network_slug = 'DMN_DAN_AMN_FPN'  # Reuse one filename-safe suffix for this selected-network set.
template = nib.load(PMTG_TEMPLATE_PATH)  # Load the pMTG CIFTI template that supplies the brain axis.
brain_axis = template.header.get_axis(1)  # Reuse the template brain axis for every label map.

selected_significant = cognitive_significant.copy()  # Work on FDR-significant cognitive EFA vertexwise rows.
if selected_significant.empty:  # Handle analyses with no FDR-significant cognitive EFA vertices.
    selected_significant['selected_display_network'] = pd.Series(dtype='object')  # Add the expected mapping column to the empty table.
else:  # Map significant rows when at least one cognitive EFA vertex survived FDR correction.
    selected_significant['selected_display_network'] = selected_significant['network'].map(  # Convert source networks to selected display labels.
        selected_display_network  # Apply the selected-network mapping function to each source network label.
    )
    selected_significant = selected_significant.dropna(subset=['selected_display_network'])  # Keep only selected-network rows.

map_measures = analysis_measures  # Create one selected-network map attempt for each cognitive EFA factor.
for measure in map_measures:  # Build one label CIFTI per cognitive EFA factor.
    factor_significant = selected_significant[selected_significant['measure'] == measure]  # Select significant vertices for this EFA factor.
    vertex_network_groups = factor_significant.groupby('python_index')['selected_display_network']  # Group selected hits by vertex index.
    vertex_networks = vertex_network_groups.apply(selected_network_tuple).to_dict()  # Convert each vertex's hits to an ordered network tuple.

    label_data = np.zeros(len(brain_axis), dtype=np.int32)  # Initialize every vertex as transparent background label 0.
    for python_index, combo in vertex_networks.items():  # Fill labels for vertices with selected-network hits.
        if combo:  # Skip empty combinations defensively.
            label_data[int(python_index)] = combo_to_label_value[combo]  # Assign the label value for this vertex's network combination.

    map_name = f'{measure}__DMN_DAN_AMN_FPN_selected_networks'  # Name the CIFTI map by EFA factor and selected-network set.
    label_axis = nib.cifti2.cifti2_axes.LabelAxis([map_name], combo_label_table)  # Create a label axis with the color table.
    header = nib.Cifti2Header.from_axes((label_axis, brain_axis))  # Combine label and brain axes into a CIFTI header.
    output_img = nib.Cifti2Image(  # Build the CIFTI label image for this factor.
        label_data[None, :],  # Add the CIFTI map dimension before the vertex axis.
        header=header,  # Attach the label/brain-axis header.
        nifti_header=template.nifti_header,  # Reuse the template NIfTI header metadata.
    )
    safe_measure = measure.replace('/', '-').replace(' ', '_')  # Make the EFA factor name safe for filenames.
    output_path = selected_factor_map_root / f'{safe_measure}__{map_network_slug}.dlabel.nii'  # Choose the output CIFTI filename.
    nib.save(output_img, output_path)  # Save the selected-network EFA factor map.

    if vertex_networks:  # Summarize observed network combinations when this factor has selected hits.
        combo_counts = pd.Series(vertex_networks).value_counts()  # Count vertices per selected-network combination.
    else:  # Create an empty count table when no selected vertices were found.
        combo_counts = pd.Series(dtype=int)  # Preserve the expected Series shape for the summary loop.
    for combo, count in combo_counts.items():  # Add one summary row per observed selected-network combination.
        selected_factor_summary_rows.append({  # Append this combination's factor-level summary record.
            'measure': measure,  # Store the residualized cognitive EFA factor name.
            'network_combo': '+'.join(combo),  # Store the selected-network combination label.
            'n_vertices': int(count),  # Store the number of vertices with this combination.
            'map_path': str(output_path),  # Store the CIFTI map path for traceability.
        })
    if not vertex_networks:  # Add an explicit zero row for factors with no selected-network hits.
        selected_factor_summary_rows.append({  # Append the empty-map summary record.
            'measure': measure,  # Store the residualized cognitive EFA factor name.
            'network_combo': 'none',  # Mark this factor as having no selected-network vertices.
            'n_vertices': 0,  # Store zero vertices for the empty map.
            'map_path': str(output_path),  # Store the CIFTI map path for traceability.
        })

selected_factor_map_summary = pd.DataFrame(selected_factor_summary_rows)  # Convert summary records to a table.
selected_factor_summary_path = selected_factor_map_root / 'selected_network_efa_factor_map_summary.csv'  # Choose the summary CSV path.
selected_factor_map_summary.to_csv(selected_factor_summary_path, index=False)  # Save the selected-network map summary.

print(f'Saved selected-network EFA factor label maps to {selected_factor_map_root}')  # Report the map output folder.
print(f'Saved selected-network EFA factor map summary to {selected_factor_summary_path}')  # Report the summary CSV path.
display(selected_factor_map_summary)  # Display the summary table in the notebook.
